In [1]:
pip install requests beautifulsoup4 selenium pandas nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\anton\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import csv
import time
import requests
import re
import pandas as pd
import unicodedata
import html
from urllib.parse import urljoin
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\anton\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anton\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
# Cabeçalhos para fingir que é navegador
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0 Safari/537.36",
    "Referer": "https://www.camara.leg.br/",
}

# URL base da pesquisa
BASE_URL = "https://www.camara.leg.br/internet/sitaqweb/resultadoPesquisaDiscursos.asp"
PARAMS = {
    "CurrentPage": 1,
    "txOrador": "",
    "txPartido": "",
    "txUF": "",
    "dtInicio": "01/01/2018",
    "dtFim": "31/01/2026",
    "txTexto":  '"Violência de gênero" OR '
                '"Proteção da mulher" OR '
                '"Proteção das mulheres" OR '
                '"Empoderamento feminino" OR '
                '"Participação feminina" OR '
                '"Delegacia da Mulher" OR '
                '"Casa da Mulher Brasileira" OR '
                '"Rede de proteção à mulher" OR '
                '"Autonomia feminina" OR '
                '"Autonomia das mulheres" OR '
                '"Representação feminina" OR '
                '"Saúde da mulher" OR '
                '"Mulheres negras" OR '
                '"Mulheres indígenas" OR '
                '"Combate ao machismo"',
            
    "txSumario": "",
    "basePesq": "plenario",
    "CampoOrdenacao": "dtSessao",
    "TipoOrdenacao": "DESC",
    "PageSize": 50,
}

# Nomes atualizados dos partidos
partidos_atualizado = {
    "PFL": "UNIAO",
    "DEM": "UNIAO",
    "PTN": "PODE",
    "PMR": "REPUBLICANOS",
    "PSL": "UNIAO",
    "PMDB": "MDB",
    "PTdoB": "AVANTE",
    "PHS": "PODE",
    "SOLIDARIEDAD": "SOLIDARIEDADE",
    "PEN": "PATRIOTA",
    "PPB": "PROGRESSISTAS",
    "PCDOB": "PCdoB"
}


# Função que monta a URL do discurso (baseado nos parâmetros da linha)
def montar_url(href):
    base = "https://www.camara.leg.br/internet/sitaqweb/"
    href_limpo = href.strip().replace("\n", "").replace("\t", "")
    return urljoin(base, href_limpo)
    query = "&".join([f"{k}={v}" for k, v in params.items()])
    return f"{base}?{query}"

# Função que extrai o texto do discurso da página HTML
def extrair_discurso(html_str):
    html_str = html_str.encode('latin-1').decode('utf-8', errors='ignore')
    soup = BeautifulSoup(html_str, "html.parser")

    hr_tag = soup.find("hr")
    texto = ""
    if hr_tag:
        for p in hr_tag.find_all_next("p"):
            t = p.get_text(" ", strip=True)
            if t:
                texto += t + " "

    # limpa quebras e espaços duplicados
    dados_unform = re.sub(r"\s+", " ", texto).strip()
    dados_unform = html.unescape(dados_unform)

    dados_format = unicodedata.normalize('NFKD', dados_unform)
    dados_format = dados_format.encode('ascii', 'ignore').decode()
    #descomentar a próxima linha caso queira tirar outros caracteres como '(',',','.','!',etc...
    #dados_format = re.sub(r'[^a-zA-Z0-9\s]', '', dados_format)

    return dados_format.strip()

# Função para atualizar partidos
def normalizar_partido(sigla):
    if not isinstance(sigla, str):
        return sigla
    sigla = sigla.strip().upper()
    return partidos_atualizado.get(sigla, sigla)

# Função para tirar stopwords
def limpar_texto(texto):
    # Tokeniza o texto
    tokens = word_tokenize(texto.lower(), language='portuguese')

    # Remove palavras irrelevantes
    tokens_filtrados = [t for t in tokens if t not in stop and len(t) > 1]

    return " ".join(tokens_filtrados)

stop = set(stopwords.words('portuguese'))

# Cria o CSV
with open("discursos_mulheres4.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Data", "Sessão", "Orador", "Partido", "Estado", "Fase", "Hora", "Discurso"])

    pagina = 1
    while True:
        print(f"Coletando página {pagina}...")

        resp = requests.get(BASE_URL, params=PARAMS, headers=HEADERS)
        if resp.status_code != 200:
            print("Erro ao acessar página, encerrando.")
            break

        soup = BeautifulSoup(resp.text, "html.parser")
        linhas = soup.select("table tbody tr")
        if not linhas:
            break

        for linha in linhas:
            colunas = linha.find_all("td")
            if len(colunas) < 6:
                continue

            data = colunas[0].get_text(strip=True)
            sessao = colunas[1].get_text(strip=True)
            
            fase_unform = colunas[2].get_text(strip=True)
            fase_unform = html.unescape(fase_unform)
            fase_unform = fase_unform.encode('latin-1').decode('utf-8', errors='ignore')
            fase_format = unicodedata.normalize('NFKD', fase_unform)
            fase_format = fase_format.encode('ascii', 'ignore').decode()

            orador = colunas[5].get_text(strip=True)
            partes = re.split(r",|-", orador)
            partes = [p.strip() for p in partes]  # limpa espaços
            if len(partes) > 3:   # corrige algumas instâncias em que aparecem dados desnecessários após o nome do orador
              i = len(partes) - 3
              while i > 0:
                del partes[i]
                i = i - 1

            orador_nome = partes[0] if len(partes) > 0 else "SEM NOME"
            orador_partido = partes[1] if len(partes) > 1 else "SEM PARTIDO"
            orador_estado = partes[2] if len(partes) > 2 else "SEM ESTADO"

            orador_nome = orador_nome.encode('latin-1').decode('utf-8', errors='ignore')
            orador_nome_format = unicodedata.normalize('NFKD', orador_nome)
            orador_nome_format = orador_nome_format.encode('ascii', 'ignore').decode()

            orador_partido_atualizado = normalizar_partido(orador_partido)

            hora = colunas[6].get_text(strip=True) if len(colunas) > 6 else ""

            # o link do discurso tem o parâmetro nuQuarto
            link_elem = colunas[3].find("a")
            if not link_elem or "href" not in link_elem.attrs:
                continue

            # extrai os parâmetros do link
            href = link_elem["href"]
            url = montar_url(href)

            try:
                r = requests.get(url, headers=HEADERS, timeout=10)
                discurso = extrair_discurso(r.text)
            except Exception as e:
                discurso = f"Erro: {e}"

            discurso = limpar_texto(discurso)

            writer.writerow([data, sessao, orador_nome_format, orador_partido_atualizado, orador_estado, fase_format, hora, discurso])
            time.sleep(0.1)

        # se não tiver botão de próxima página, para
        next_page = soup.find("a", string=lambda t: t and "Próxima" in t)
        if not next_page:
            break
        
        pagina += 1
        PARAMS["CurrentPage"] = pagina

print("Finalizado. Arquivo salvo como discursos_mulheres1.csv")

Coletando página 1...
Coletando página 2...
Coletando página 3...
Coletando página 4...
Coletando página 5...
Coletando página 6...
Coletando página 7...
Coletando página 8...
Coletando página 9...
Coletando página 10...
Coletando página 11...
Coletando página 12...
Coletando página 13...
Coletando página 14...
Coletando página 15...
Coletando página 16...
Coletando página 17...
Coletando página 18...
Coletando página 19...
Finalizado. Arquivo salvo como discursos_mulheres1.csv


In [ ]:
BASE_URL = "https://www.camara.leg.br/internet/sitaqweb/resultadoPesquisaDiscursos.asp"
PARAMS = {
    "txOrador": "",
    "txPartido": "",
    "txUF": "",
    "dtInicio": "01/01/2000",
    "dtFim": "31/01/2026",
    "txTexto":  '"Violência contra a mulher" OR '
                '"Violência contra as mulheres" OR'
                '"Violência doméstica" OR '
                '"Lei Maria da Penha" OR '
                'Feminicídio OR '
                'Feminicídios OR '
                '"Igualdade de gênero" OR '
                '"Igualdade de generos" OR '
                '"Equidade de gênero" OR '
                '"Equidade de generos" OR '
                '"Políticas para mulheres" OR '
                '"Direitos das mulheres" OR '
                '"Direitos da mulher" OR '
                'Machismo OR '
                'Misoginia OR '
                'Sexismo OR '
                '"Violência sexual" OR '
                'Estupro OR '
                '"Assédio sexual"',
                #------------------------------------
                '"Assédio contra mulheres" OR '
                '"Igualdade salarial" OR '
                '"Equidade salarial" OR '
                'Feminismo',
                #------------------------------------
                '"Violência de gênero" OR '
                '"Proteção da mulher" OR '
                '"Proteção das mulheres" OR '
                '"Empoderamento feminino" OR '
                '"Participação feminina" OR '
                '"Delegacia da Mulher" OR '
                '"Casa da Mulher Brasileira" OR '
                '"Rede de proteção à mulher" OR '
                '"Autonomia feminina" OR '
                '"Autonomia das mulheres" OR '
                '"Representação feminina" OR '
                '"Saúde da mulher" OR '
                '"Mulheres negras" OR '
                '"Mulheres indígenas" OR '
                '"Combate ao machismo"',
    "txSumario": "",
    "basePesq": "plenario",
    "CampoOrdenacao": "dtSessao",
    "TipoOrdenacao": "DESC",
    "PageSize": 50,
}
resp = requests.get(BASE_URL, params=PARAMS)
print(resp.url)

https://www.camara.leg.br/internet/sitaqweb/resultadoPesquisaDiscursos.asp?txOrador=&txPartido=&txUF=&dtInicio=01%2F01%2F2000&dtFim=31%2F01%2F2026&txTexto=%22Equidade+salarial%22+OR+Feminismo&txSumario=&basePesq=plenario&CampoOrdenacao=dtSessao&TipoOrdenacao=DESC&PageSize=50


In [11]:
import glob

# carrega todos os CSVs de uma vez
arquivos = glob.glob("discursos_mulheres*.csv")  # ajusta o padrão pro nome dos seus arquivos
dfs = [pd.read_csv(f, encoding="utf-8") for f in arquivos]

# junta tudo
df_total = pd.concat(dfs, ignore_index=True)

print(f"Total antes de deduplicar: {len(df_total)}")

# remove duplicatas — ajusta as colunas conforme seu caso
df_total = df_total.drop_duplicates(subset=["Data", "Orador", "Hora"])

print(f"Total depois de deduplicar: {len(df_total)}")

df_total.to_csv("discursos_completo2.csv", index=False, encoding="utf-8")
print("Salvo como discursos_completo2.csv")

Total antes de deduplicar: 4107
Total depois de deduplicar: 3628
Salvo como discursos_completo2.csv
